# S-DoT 데이터 전처리 및 UTCI 계산

**목적**: S-DoT 환경센서 데이터(2025.07.28~08.03)를 이용해 성동구 내 센서 위치별 시간대별 UTCI를 계산한다.

## 입력
- `S-DoT_NATURE_2025.07.28-08.03/*.csv` — 7일치 환경데이터
- `서울시 도시데이터 센서(S-DoT) 환경정보 설치 위치정보.xlsx` — 센서 위도/경도

## 출력
- `../04_분석결과/sdot_utci_seongdong.csv` — 성동구 센서별 시간대별 UTCI

## UTCI 계산 방식
| 변수 | 출처 | 비고 |
|---|---|---|
| tdb (기온) | S-DoT avg_temp | 실측 |
| rh (습도) | S-DoT avg_humi | 실측 |
| v (풍속) | **1.5 m/s 가정** | 추후 ASOS 교체 예정 |
| tr (MRT) | tdb + 3.5 × UV_index | UV=0 야간엔 tr=tdb, 낮엔 +최대 13°C |

> **한계 명시**: tr은 S-DoT UV 지수로 추정한 1차 근사값. 건물 그늘/SVF는 미반영.

In [ ]:
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from pythermalcomfort.models import utci
from pathlib import Path

DATA_DIR = Path('S-DoT_NATURE_2025.07.28-08.03')
LOC_FILE = Path('서울시 도시데이터 센서(S-DoT) 환경정보 설치 위치정보.xlsx')
OUT_DIR  = Path('../04_분석결과')
OUT_DIR.mkdir(exist_ok=True)

V_ASSUMED = 1.5   # 풍속 가정값 (m/s) — 추후 ASOS 값으로 교체
K_UV      = 3.5   # MRT = tdb + K_UV × UV_index

print('설정 완료')

## 1. 위치정보 로드

In [ ]:
loc = pd.read_excel(LOC_FILE)
loc = loc.rename(columns={'모델 시리얼(*)': 'serial', '위도': 'lat', '경도': 'lon', '주소': 'address'})
loc = loc[['serial', 'lat', 'lon', 'address']].dropna(subset=['lat', 'lon'])

# 성동구 필터
loc_sdong = loc[loc['address'].str.contains('성동구', na=False)].copy()
print(f'성동구 센서 위치정보: {len(loc_sdong)}개')
loc_sdong.head()

## 2. S-DoT CSV 7일치 로드 + 성동구 필터

In [ ]:
files = sorted(glob.glob(str(DATA_DIR / '*.csv')))
print(f'파일 수: {len(files)}')

dfs = []
for f in files:
    df = pd.read_csv(f, encoding='cp949')
    df = df[df['autonomous_district'].str.contains('Seongdong', na=False)]
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)
print(f'성동구 전체 레코드: {len(raw):,}행 / 센서 {raw["시리얼"].nunique()}개')
raw.head(2)

## 3. 컬럼 정리 + 시간 파싱

In [ ]:
df = raw.rename(columns={
    '시리얼': 'serial',
    'sensing_time': 'time_raw',
    'avg_temp (℃)': 'temp',
    'avg_humi (%)': 'humi',
    'avg_ultra_rays (UV)': 'uv',
    'avg_effe_temp (℃)': 'effe_temp',
    'administrative_district': 'dong'
})[['serial', 'time_raw', 'dong', 'temp', 'humi', 'uv', 'effe_temp']].copy()

# 시간 파싱: '2025-07-28_00:07:00' → datetime
df['dt'] = pd.to_datetime(df['time_raw'].str.replace('_', ' '), errors='coerce')
df['hour'] = df['dt'].dt.hour
df['date'] = df['dt'].dt.date

# 결측 제거 (temp, humi 필수)
df = df.dropna(subset=['temp', 'humi'])
print(f'유효 레코드: {len(df):,}행')
print(f'날짜 범위: {df["date"].min()} ~ {df["date"].max()}')
df.head(3)

## 4. MRT 추정 + UTCI 계산

In [ ]:
# UV 결측은 0으로 처리 (야간 = 태양복사 없음)
df['uv'] = df['uv'].fillna(0)

# MRT 추정: tr = tdb + 3.5 × UV_index
df['tr_est'] = df['temp'] + K_UV * df['uv']

# 풍속 가정
df['v'] = V_ASSUMED

# UTCI 계산 (pythermalcomfort) — result.utci로 값 접근
def calc_utci(row):
    try:
        result = utci(tdb=row['temp'], tr=row['tr_est'], v=row['v'], rh=row['humi'])
        return float(result.utci)
    except Exception:
        return np.nan

df['utci_val'] = df.apply(calc_utci, axis=1)

# UTCI 열 스트레스 등급
def utci_category(u):
    if pd.isna(u):   return 'unknown'
    if u < 9:        return 'cold'
    if u < 26:       return 'comfortable'
    if u < 32:       return 'moderate_heat'
    if u < 38:       return 'strong_heat'
    if u < 46:       return 'very_strong_heat'
    return 'extreme_heat'

df['utci_cat'] = df['utci_val'].apply(utci_category)

print(f'UTCI 계산 완료: {df["utci_val"].notna().sum():,}건')
df[['serial', 'dt', 'hour', 'temp', 'tr_est', 'humi', 'utci_val', 'utci_cat']].head(5)

## 5. 위치정보 조인 → GeoDataFrame

In [ ]:
# 시리얼 기준 조인
merged = df.merge(loc_sdong[['serial', 'lat', 'lon', 'address']], on='serial', how='inner')
print(f'위치 조인 후: {len(merged):,}행 / 센서 {merged["serial"].nunique()}개')

# GeoDataFrame 변환
gdf = gpd.GeoDataFrame(
    merged,
    geometry=gpd.points_from_xy(merged['lon'], merged['lat']),
    crs='EPSG:4326'
).to_crs('EPSG:5186')

print(f'좌표계: EPSG:5186 변환 완료')
gdf[['serial', 'dt', 'hour', 'lat', 'lon', 'utci_val', 'utci_cat']].head(3)

## 6. 시간대별 UTCI 분포 확인

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'  # Mac 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False

hourly = gdf.groupby('hour')['utci_val'].agg(['mean', 'min', 'max'])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(hourly.index, hourly['mean'], 'o-', color='crimson', linewidth=2, label='평균 UTCI')
ax.fill_between(hourly.index, hourly['min'], hourly['max'], alpha=0.15, color='crimson', label='센서간 범위')

# 열 스트레스 기준선
ax.axhline(26, color='orange', linestyle='--', linewidth=0.8, label='약한 열(26°C)')
ax.axhline(32, color='red',    linestyle='--', linewidth=0.8, label='강한 열(32°C)')
ax.axhline(38, color='darkred',linestyle='--', linewidth=0.8, label='매우 강한 열(38°C)')

ax.set_xlabel('시각 (시)')
ax.set_ylabel('UTCI (°C)')
ax.set_title('성동구 S-DoT 센서 시간대별 UTCI (2025.07.28~08.03 평균)')
ax.set_xticks(range(0, 24))
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../05_시각화/sdot_utci_hourly.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 05_시각화/sdot_utci_hourly.png')

## 7. 폭염 피크 시간대(13시) 공간 분포 확인

In [ ]:
peak = gdf[gdf['hour'] == 13].copy()
peak_avg = peak.groupby('serial').agg(
    lat=('lat', 'first'), lon=('lon', 'first'),
    utci_mean=('utci_val', 'mean'),
    geometry=('geometry', 'first')
).reset_index()
peak_gdf = gpd.GeoDataFrame(peak_avg, geometry='geometry', crs='EPSG:5186')

fig, ax = plt.subplots(figsize=(10, 10))
peak_gdf.plot(
    column='utci_mean', ax=ax,
    cmap='RdYlGn_r', markersize=80,
    legend=True,
    legend_kwds={'label': 'UTCI (°C)', 'shrink': 0.6}
)
ax.set_title('성동구 폭염 피크(13시) 평균 UTCI 공간 분포')
ax.set_axis_off()
plt.tight_layout()
plt.savefig('../05_시각화/sdot_utci_peak13h_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: 05_시각화/sdot_utci_peak13h_map.png')

## 8. 결과 저장

In [ ]:
# CSV 저장 (링크 보간 단계에서 사용)
out_cols = ['serial', 'dt', 'date', 'hour', 'dong', 'lat', 'lon',
            'temp', 'humi', 'uv', 'tr_est', 'v', 'utci_val', 'utci_cat']
save_df = merged[out_cols].copy()
save_df.to_csv(OUT_DIR / 'sdot_utci_seongdong.csv', index=False, encoding='utf-8-sig')

print(f'저장 완료: 04_분석결과/sdot_utci_seongdong.csv')
print(f'  총 {len(save_df):,}행 / 센서 {save_df["serial"].nunique()}개 / {save_df["date"].nunique()}일')

# 시간대별 요약 통계
summary = save_df.groupby('hour')['utci_val'].agg(['mean','min','max','count']).round(2)
print('\n시간대별 UTCI 요약:')
print(summary.to_string())